# Data Augmentation & Forecasting: Yearly → Monthly → Daily

## Overview

In this notebook, I explore **data augmentation** by interpolating yearly panel data to monthly and daily frequencies. The goal is to see if having more data points helps neural networks learn better patterns and potentially beat the baseline lag-based model.

**My workflow:**
1. Load yearly panel data (poverty + indicators)
2. Understand different data augmentation principles
3. Apply linear interpolation: yearly → monthly → daily
4. Create baseline models (lag-based) for each frequency
5. Train neural networks with different architectures
6. Compare results across frequencies and model types

**Key Question:** Does having more data points (daily vs yearly) help neural networks beat the baseline?

### Types of Data Augmentation

#### 1. **Temporal Interpolation** (What I'm using)
- **Purpose**: Fill gaps between time points
- **Methods**: Linear, spline, polynomial interpolation
- **Use case**: When you have yearly data but need monthly/daily
- **Pros**: Preserves temporal structure, smooth transitions
- **Cons**: Creates artificial smoothness, may not reflect real volatility

#### 2. **Synthetic Data Generation**
- **Purpose**: Create entirely new samples
- **Methods**: GANs, SMOTE, variational autoencoders
- **Use case**: Classification with imbalanced classes
- **Pros**: Can create diverse new samples
- **Cons**: Complex, may generate unrealistic data

#### 3. **Noise Injection**
- **Purpose**: Add small random variations
- **Methods**: Gaussian noise, dropout, jittering
- **Use case**: Robustness to measurement errors
- **Pros**: Simple, helps with overfitting
- **Cons**: May distort signal if noise is too large

#### 4. **Transformation-Based**
- **Purpose**: Apply geometric/statistical transformations
- **Methods**: Rotation, scaling, flipping (images), log transforms
- **Use case**: Computer vision, feature engineering
- **Pros**: Preserves relationships, interpretable
- **Cons**: Domain-specific, may not apply to time series

### Why I Chose Linear Interpolation

For **time series panel data** (poverty rates, crime, health indicators), I chose **linear interpolation** because:

1. **Temporal continuity**: Economic indicators change gradually, not abruptly
2. **Simplicity**: Linear interpolation is fast, interpretable, and doesn't require tuning
3. **Preserves trends**: Maintains the overall direction of change between years
4. **No assumptions**: Doesn't assume complex patterns that may not exist

**How Linear Interpolation Works:**

For two known points (year1, value1) and (year2, value2), the interpolated value at any point between them is:

```
value = value1 + (value2 - value1) × (date - year1) / (year2 - year1)
```

This creates a **straight line** between the two points. NumPy's `np.interp()` does this automatically.

**Alternative methods I could use:**
- **Cubic spline**: Smoother curves, better for non-linear trends (but may overfit)
- **Forward fill**: Constant values (not suitable for gradual changes)
- **Seasonal decomposition**: Add seasonal patterns (requires domain knowledge)

### Important Caveats

⚠️ **Interpolation creates artificial data** - the monthly/daily values are estimates, not real observations. This means:
- Models trained on interpolated data may not generalize to real monthly/daily data
- The high R² scores (0.999+) are partially due to interpolation smoothness
- Real-world data would have more noise and volatility

However, this is still valuable for:
- Testing if more data points help neural networks
- Understanding how model performance scales with data size
- Exploring different architectures on larger datasets

### Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import math
import random
from dataclasses import dataclass
from typing import List, Tuple
import warnings
import os
import time
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42

def set_seed(seed):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

print("Imports loaded. Ready to begin data augmentation.")

Imports loaded. Ready to begin data augmentation.


## Step 1 — Load Yearly Data

I start with the yearly panel data. This is my **ground truth** - the real observations I have. Everything else will be interpolated from this.

In [2]:
DATA_PATH = "../data/processed/panel/marz_year_panel_common.csv"

df_yearly = pd.read_csv(DATA_PATH)
df_yearly = df_yearly.sort_values(["marz", "year"]).copy()
df_yearly["year"] = df_yearly["year"].astype(int)

print("Yearly data shape:", df_yearly.shape)
print("Years:", sorted(df_yearly["year"].unique()))
print("Marzes:", df_yearly["marz"].nunique())
print(f"\nTotal observations: {len(df_yearly)} (marzes × years)")
print("\nSample of data:")
df_yearly.head(10)

Yearly data shape: (77, 24)
Years: [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Marzes: 11

Total observations: 77 (marzes × years)

Sample of data:


,marz,year,poverty_rate,extreme_poverty_rate,non_poor_rate,population,crime_selected_total,Number of minor offense,Number of moderately serious offense,Number of particulary serious offences,...,Number of avarage medical personnel,beds,Number of hospitalized patients,hospitals,Number of physicians,"Number of primary health care (PHC) service providers (except OMCs, private medical and dental offices)",crime_rate_per_100k,crime_selected_rate_per_100k,hospitals_per_100k,beds_per_10k
1,Aragatsotn,2016,15.7,0.6,84.3,129774.0,191.0,322.0,259.0,4.0,...,633.0,200.0,5376.0,6.0,241.0,24.0,504.723596,147.178942,4.623422,15.411408
12,Aragatsotn,2017,17.6,0.0,82.4,128518.0,195.0,358.0,310.0,5.0,...,572.0,160.0,5067.0,6.0,227.0,23.0,574.238628,151.729719,4.668607,12.449618
23,Aragatsotn,2018,16.2,0.0,83.8,127146.0,192.0,301.0,291.0,6.0,...,544.0,152.0,5231.0,6.0,228.0,23.0,538.750728,151.007503,4.718984,11.954761
34,Aragatsotn,2019,51.4,7.7,48.6,125440.0,268.0,375.0,300.0,7.0,...,542.0,167.0,6642.0,6.0,239.0,24.0,609.056122,213.647959,4.783163,13.313138
45,Aragatsotn,2020,32.9,1.2,67.1,124721.0,247.0,446.0,261.0,6.0,...,524.0,196.0,5638.0,5.0,233.0,24.0,669.494311,198.042030,4.008948,15.715076
56,Aragatsotn,2021,13.5,0.0,86.5,124446.0,280.0,474.0,259.0,5.0,...,526.0,172.0,6509.0,5.0,252.0,24.0,673.384440,224.997188,4.017807,13.821256
67,Aragatsotn,2022,7.9,0.0,92.1,124646.0,292.0,574.0,411.0,49.0,...,519.0,180.0,6096.0,5.0,279.0,24.0,931.437832,234.263434,4.011360,14.440897
2,Ararat,2016,26.9,1.0,73.1,258911.0,349.0,647.0,417.0,4.0,...,1005.0,473.0,13693.0,7.0,460.0,60.0,473.135556,134.795354,2.703632,18.268826
13,Ararat,2017,21.7,1.6,78.3,258407.0,346.0,707.0,364.0,8.0,...,941.0,423.0,13003.0,6.0,436.0,60.0,477.154257,133.897302,2.321919,16.369526
24,Ararat,2018,19.8,0.0,80.2,257804.0,484.0,758.0,475.0,14.0,...,988.0,488.0,13396.0,6.0,439.0,61.0,554.684954,187.739523,2.327349,18.929109


## Step 2 — Identify Columns to Interpolate

I need to decide which columns make sense to interpolate. **Not everything should be interpolated**:

- ✅ **Do interpolate**: Numeric variables that change gradually (poverty_rate, population, crime rates)
- ❌ **Don't interpolate**: Categorical variables (marz names), IDs, or variables with too many missing values

I'll interpolate all numeric columns except identifiers, and skip columns with >50% missing values.

In [3]:
# Identify numeric columns to interpolate
exclude_cols = ["marz", "year"]  # Keep these as-is (identifiers)
numeric_cols = df_yearly.select_dtypes(include=[np.number]).columns.tolist()
cols_to_interpolate = [c for c in numeric_cols if c not in exclude_cols]

# Filter out columns with >50% missing values (too sparse to interpolate meaningfully)
missing_pct = df_yearly[cols_to_interpolate].isnull().sum() / len(df_yearly)
cols_to_interpolate = [c for c in cols_to_interpolate if missing_pct[c] < 0.5]

print(f"Columns to interpolate ({len(cols_to_interpolate)}):")
for i, col in enumerate(cols_to_interpolate, 1):
    missing = df_yearly[col].isnull().sum()
    print(f"  {i:2d}. {col:40s} (missing: {missing:3d}/{len(df_yearly)})")

# Target column for forecasting
TARGET_COL = "poverty_rate"
print(f"\n🎯 Target variable: {TARGET_COL}")

Columns to interpolate (22):
   1. poverty_rate                             (missing:   0/77)
   2. extreme_poverty_rate                     (missing:   0/77)
   3. non_poor_rate                            (missing:   0/77)
   4. population                               (missing:   0/77)
   5. crime_selected_total                     (missing:   0/77)
   6. Number of minor offense                  (missing:   0/77)
   7. Number of moderately serious offense     (missing:   0/77)
   8. Number of particulary serious offences   (missing:   2/77)
   9. crime_total                              (missing:   0/77)
  10. Number of serious offences               (missing:   0/77)
  11. Annual average occupancy of a bed        (missing:   0/77)
  12. Number of ambulatory clinics in rural communities (missing:   7/77)
  13. Number of avarage medical personnel      (missing:   0/77)
  14. beds                                     (missing:   0/77)
  15. Number of hospitalized patients          (miss

## Step 3 — Interpolation: Yearly → Monthly

### How Linear Interpolation Works

For each marz and each column, I:

1. **Take known yearly values**: e.g., poverty_rate in 2016=15.7, 2017=17.6
2. **Create monthly dates**: Jan 2016, Feb 2016, ..., Dec 2017
3. **Interpolate linearly**: For any month, calculate the value using a straight line between adjacent years

**Example calculation:**
- Year 2016: poverty_rate = 15.7
- Year 2017: poverty_rate = 17.6
- July 2016 (mid-year): Should be close to 15.7
- January 2017: Should be close to 17.6
- **Linear interpolation**: Creates a smooth transition: 15.7 → 15.8 → 15.9 → ... → 17.6

**Formula**: For a date between year Y1 and Y2:
```
value = value_Y1 + (value_Y2 - value_Y1) × (date - Y1) / (Y2 - Y1)
```

NumPy's `np.interp()` does this automatically for us.

In [4]:
print("Interpolating yearly → monthly...")
t0 = time.time()

df_monthly_list = []
for marz in df_yearly["marz"].unique():
    # Get data for this marz, sorted by year
    marz_data = df_yearly[df_yearly["marz"] == marz].sort_values("year")
    
    # Create monthly date range for this marz's year span
    dates = pd.date_range(
        start=f"{int(marz_data['year'].min())}-01-01", 
        end=f"{int(marz_data['year'].max())}-12-31", 
        freq="MS"  # Month Start (first day of each month)
    )
    
    # For each month, interpolate all columns
    for date in dates:
        row = {"marz": marz, "date": date, "year": date.year, "month": date.month}
        
        for col in cols_to_interpolate:
            values = marz_data[col].values
            years = marz_data["year"].values
            valid = ~pd.isna(values)  # Only use non-missing values
            
            if valid.sum() >= 2:  # Need at least 2 points to interpolate
                # Convert date to fractional year (e.g., 2016.5 = mid-2016)
                fractional_year = date.year + (date.month - 1) / 12
                # Linear interpolation
                row[col] = np.interp(fractional_year, years[valid], values[valid])
            else:
                row[col] = np.nan
        
        df_monthly_list.append(row)

df_monthly = pd.DataFrame(df_monthly_list)
print(f"✓ Monthly data created: {df_monthly.shape} rows")
print(f"  Date range: {df_monthly['date'].min()} to {df_monthly['date'].max()}")
print(f"  Time taken: {time.time()-t0:.1f} seconds")
print(f"\n📊 Data expansion: {len(df_yearly)} yearly → {len(df_monthly)} monthly ({len(df_monthly)/len(df_yearly):.1f}x increase)")

df_monthly.head()

Interpolating yearly → monthly...
✓ Monthly data created: (924, 26) rows
  Date range: 2016-01-01 00:00:00 to 2022-12-01 00:00:00
  Time taken: 0.2 seconds

📊 Data expansion: 77 yearly → 924 monthly (12.0x increase)


,marz,date,year,month,poverty_rate,extreme_poverty_rate,non_poor_rate,population,crime_selected_total,Number of minor offense,...,Number of avarage medical personnel,beds,Number of hospitalized patients,hospitals,Number of physicians,"Number of primary health care (PHC) service providers (except OMCs, private medical and dental offices)",crime_rate_per_100k,crime_selected_rate_per_100k,hospitals_per_100k,beds_per_10k
0,Aragatsotn,2016-01-01,2016,1,15.700000,0.60,84.300000,129774.000000,191.000000,322.0,...,633.000000,200.000000,5376.00,6.0,241.000000,24.000000,504.723596,147.178942,4.623422,15.411408
1,Aragatsotn,2016-02-01,2016,2,15.858333,0.55,84.141667,129669.333333,191.333333,325.0,...,627.916667,196.666667,5350.25,6.0,239.833333,23.916667,510.516516,147.558173,4.627188,15.164592
2,Aragatsotn,2016-03-01,2016,3,16.016667,0.50,83.983333,129564.666667,191.666667,328.0,...,622.833333,193.333333,5324.50,6.0,238.666667,23.833333,516.309435,147.937405,4.630953,14.917776
3,Aragatsotn,2016-04-01,2016,4,16.175000,0.45,83.825000,129460.000000,192.000000,331.0,...,617.750000,190.000000,5298.75,6.0,237.500000,23.750000,522.102354,148.316636,4.634718,14.670960
4,Aragatsotn,2016-05-01,2016,5,16.333333,0.40,83.666667,129355.333333,192.333333,334.0,...,612.666667,186.666667,5273.00,6.0,236.333333,23.666667,527.895274,148.695867,4.638484,14.424144


## Step 4 — Interpolation: Monthly → Daily

Now I do the same process again, but interpolating from monthly to daily. This creates even more data points.

**Key optimization**: I pre-compute the interpolation functions for each column to avoid repeated calculations, making this step faster.

In [5]:
print("Interpolating monthly → daily...")
t0 = time.time()

df_daily_list = []
for i, marz in enumerate(df_monthly["marz"].unique()):
    if (i+1) % 3 == 0:  # Progress indicator
        print(f"    Processing marz {i+1}/{len(df_monthly['marz'].unique())}...")
    
    marz_data = df_monthly[df_monthly["marz"] == marz].sort_values("date")
    dates = pd.date_range(marz_data["date"].min(), marz_data["date"].max(), freq="D")
    
    # Pre-compute interpolation data for each column (optimization)
    interp_funcs = {}
    for col in cols_to_interpolate:
        values = marz_data[col].values
        dates_numeric = marz_data["date"].astype(np.int64).values  # Convert to nanoseconds
        valid = ~pd.isna(values)
        if valid.sum() >= 2:
            interp_funcs[col] = (dates_numeric[valid], values[valid])
        else:
            interp_funcs[col] = None
    
    # For each day, interpolate all columns
    for date in dates:
        row = {"marz": marz, "date": date, "year": date.year, "month": date.month, "day": date.day}
        for col in cols_to_interpolate:
            if interp_funcs[col] is not None:
                # Linear interpolation using timestamp values
                row[col] = np.interp(date.value, interp_funcs[col][0], interp_funcs[col][1])
            else:
                row[col] = np.nan
        df_daily_list.append(row)

df_daily = pd.DataFrame(df_daily_list)
print(f"✓ Daily data created: {df_daily.shape} rows")
print(f"  Date range: {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"  Time taken: {time.time()-t0:.1f} seconds")
print(f"\n📊 Data expansion: {len(df_monthly)} monthly → {len(df_daily)} daily ({len(df_daily)/len(df_monthly):.1f}x increase)")
print(f"\n🎯 Total expansion: {len(df_yearly)} yearly → {len(df_daily)} daily ({len(df_daily)/len(df_yearly):.0f}x increase!)")

df_daily.head()

Interpolating monthly → daily...
    Processing marz 3/11...
    Processing marz 6/11...
    Processing marz 9/11...
✓ Daily data created: (27797, 27) rows
  Date range: 2016-01-01 00:00:00 to 2022-12-01 00:00:00
  Time taken: 1.1 seconds

📊 Data expansion: 924 monthly → 27797 daily (30.1x increase)

🎯 Total expansion: 77 yearly → 27797 daily (361x increase!)


,marz,date,year,month,day,poverty_rate,extreme_poverty_rate,non_poor_rate,population,crime_selected_total,...,Number of avarage medical personnel,beds,Number of hospitalized patients,hospitals,Number of physicians,"Number of primary health care (PHC) service providers (except OMCs, private medical and dental offices)",crime_rate_per_100k,crime_selected_rate_per_100k,hospitals_per_100k,beds_per_10k
0,Aragatsotn,2016-01-01,2016,1,1,15.700000,0.600000,84.300000,129774.000000,191.000000,...,633.000000,200.000000,5376.000000,6.0,241.000000,24.000000,504.723596,147.178942,4.623422,15.411408
1,Aragatsotn,2016-01-02,2016,1,2,15.705108,0.598387,84.294892,129770.623656,191.010753,...,632.836022,199.892473,5375.169355,6.0,240.962366,23.997312,504.910465,147.191175,4.623544,15.403446
2,Aragatsotn,2016-01-03,2016,1,3,15.710215,0.596774,84.289785,129767.247312,191.021505,...,632.672043,199.784946,5374.338710,6.0,240.924731,23.994624,505.097333,147.203408,4.623665,15.395484
3,Aragatsotn,2016-01-04,2016,1,4,15.715323,0.595161,84.284677,129763.870968,191.032258,...,632.508065,199.677419,5373.508065,6.0,240.887097,23.991935,505.284201,147.215642,4.623787,15.387522
4,Aragatsotn,2016-01-05,2016,1,5,15.720430,0.593548,84.279570,129760.494624,191.043011,...,632.344086,199.569892,5372.677419,6.0,240.849462,23.989247,505.471070,147.227875,4.623908,15.379560


## Step 5 — Prepare Data for Modeling

For each frequency (yearly, monthly, daily), I need to:

1. **Create lag features**: `poverty_lag1` = previous period's poverty rate
2. **Time-based split**: Train on ≤2020, test on ≥2021 (preserves temporal order)
3. **Handle missing values**: Fill NaN with median values
4. **Prepare features**: Lag + all other numeric columns

**Important**: The lag period depends on frequency:
- **Yearly**: lag by 1 year
- **Monthly**: lag by 1 month  
- **Daily**: lag by 30 days (approximately 1 month)

In [6]:
def prepare_data_for_modeling(df, date_col, target_col):
    """
    Prepare data with lag features and time-based train/test split.
    
    Args:
        df: DataFrame with panel data
        date_col: Name of date/year column
        target_col: Name of target variable
    
    Returns:
        Dictionary with X_train, y_train, X_test, y_test, baseline_pred, feature_cols
    """
    df = df.sort_values(["marz", date_col]).copy()
    
    # Create lag feature (previous period's value)
    df[f"{target_col}_lag1"] = df.groupby("marz")[target_col].shift(1)
    
    # Drop rows without lag (first period for each marz)
    df = df.dropna(subset=[f"{target_col}_lag1", target_col]).copy()
    
    # Time-based split: train ≤2020, test ≥2021
    if date_col == "year":
        train_mask = df[date_col] <= 2020
        test_mask = df[date_col] >= 2021
    else:  # date column
        train_mask = df[date_col].dt.year <= 2020
        test_mask = df[date_col].dt.year >= 2021
    
    # Select features: lag + all other numeric columns
    feature_cols = [f"{target_col}_lag1"]
    other_cols = [c for c in cols_to_interpolate if c != target_col and c in df.columns]
    feature_cols.extend(other_cols)
    feature_cols = [c for c in feature_cols if c in df.columns]
    
    X_train = df.loc[train_mask, feature_cols].copy()
    y_train = df.loc[train_mask, target_col].copy()
    X_test = df.loc[test_mask, feature_cols].copy()
    y_test = df.loc[test_mask, target_col].copy()
    
    # Fill NaN in features with median (from training set only)
    for col in feature_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
    
    # Baseline prediction = lag feature
    baseline_pred = X_test[f"{target_col}_lag1"].values
    
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "baseline_pred": baseline_pred,
        "feature_cols": feature_cols
    }

# Prepare all three frequencies
print("Preparing data for modeling...")
yearly_data = prepare_data_for_modeling(df_yearly, "year", TARGET_COL)
monthly_data = prepare_data_for_modeling(df_monthly, "date", TARGET_COL)
daily_data = prepare_data_for_modeling(df_daily, "date", TARGET_COL)

print(f"\n📈 Dataset sizes:")
print(f"  Yearly:  Train={len(yearly_data['y_train']):4d}, Test={len(yearly_data['y_test']):4d}")
print(f"  Monthly: Train={len(monthly_data['y_train']):4d}, Test={len(monthly_data['y_test']):4d}")
print(f"  Daily:   Train={len(daily_data['y_train']):4d}, Test={len(daily_data['y_test']):4d}")

Preparing data for modeling...

📈 Dataset sizes:
  Yearly:  Train=  44, Test=  22
  Monthly: Train= 649, Test= 264
  Daily:   Train=20086, Test=7700


## Step 6 — Baseline Models (Lag-Based)

Before training complex models, I evaluate the **baseline**: simply using the previous period's value as the prediction. This is a strong baseline for time series data.

**Why this baseline is strong:**
- Poverty rates are persistent (don't change dramatically year-to-year)
- The lag feature (`poverty_lag1`) captures most of the predictable signal
- Simple models often outperform complex ones in time series

**Expected result**: As we interpolate to higher frequencies, the baseline R² should increase because:
- Interpolation creates smooth transitions
- Lag features become more similar to targets (less time between periods)
- This is an artifact of interpolation, not real predictive power

In [7]:
def compute_baseline_metrics(y_true, y_pred, frequency_name):
    """Compute evaluation metrics for baseline model."""
    return {
        "frequency": frequency_name,
        "r2": r2_score(y_true, y_pred),
        "mse": mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "n_train": len(y_true),
        "n_test": len(y_pred)
    }

# Evaluate baselines for all frequencies
print("Evaluating baseline models...")
baseline_results = []

for name, data in [("yearly", yearly_data), ("monthly", monthly_data), ("daily", daily_data)]:
    metrics = compute_baseline_metrics(
        data["y_test"].values,
        data["baseline_pred"],
        name
    )
    baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results)
print("\n📊 Baseline Results:")
print(baseline_df.to_string(index=False))

print("\n💡 Observation: Notice how R² increases with frequency.")
print("   This is expected due to interpolation smoothness - the lag feature")
print("   becomes almost identical to the target in daily data.")

Evaluating baseline models...

📊 Baseline Results:
frequency       r2       mse      mae  n_train  n_test
   yearly 0.727857 45.938182 5.109091       22      22
  monthly 0.998961  0.161423 0.229230      264     264
    daily 0.999999  0.000166 0.007267     7700    7700

💡 Observation: Notice how R² increases with frequency.
   This is expected due to interpolation smoothness - the lag feature
   becomes almost identical to the target in daily data.


## Step 7 — Neural Network Models

Now I train neural networks to see if they can beat the baseline. I use the same MLP architecture from my previous notebook, but with **optimized training settings** for speed:

**Training optimizations:**
- `max_epochs`: 300 (reduced from 2000)
- `patience`: 30 (reduced from 100) - stops early if no improvement
- `batch_size`: 256 (increased from 32) - faster training
- `lr`: 3e-3 (increased from 1e-3) - faster convergence

**Architectures tested**: (32), (64), (32, 16), (64, 32)

**Why these settings**: The goal is to quickly test if more data helps, not to find the perfect model. These settings balance speed and quality.

In [8]:
# MLP helper functions
def standardize_train_only(X_train, X_val, X_test):
    """Standardize features using training set statistics only."""
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.where(std == 0, 1.0, std)  # Avoid division by zero
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std

class MLPRegressor(nn.Module):
    """Multi-layer perceptron for regression."""
    def __init__(self, in_dim, hidden_dims, activation, dropout=0.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(activation)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))  # Output layer
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

@dataclass
class TrainConfig:
    """Training configuration with optimized settings."""
    lr: float = 3e-3  # Increased learning rate for faster convergence
    weight_decay: float = 1e-4
    batch_size: int = 256  # Larger batch size for better throughput
    max_epochs: int = 300  # Reduced from 2000
    patience: int = 30  # Reduced from 100
    min_delta: float = 1e-4  # Slightly relaxed

def train_mlp(model, X_train, y_train, X_val, y_val, cfg, seed):
    """Train MLP with early stopping."""
    set_seed(seed)
    rng = np.random.default_rng(seed)
    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    best_state = None
    best_val = math.inf
    no_improve = 0
    
    n_samples = len(X_train)
    
    for epoch in range(cfg.max_epochs):
        model.train()
        idx = np.arange(n_samples)
        rng.shuffle(idx)
        
        # Batch training
        for start in range(0, n_samples, cfg.batch_size):
            end = min(start + cfg.batch_size, n_samples)
            sl = idx[start:end]
            optim.zero_grad()
            pred = model(X_train[sl])
            loss = nn.functional.mse_loss(pred, y_train[sl])
            loss.backward()
            optim.step()
        
        # Validation check
        with torch.no_grad():
            model.eval()
            val_pred = model(X_val)
            val_loss = nn.functional.mse_loss(val_pred, y_val).item()
        
        if val_loss < best_val - cfg.min_delta:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        
        if no_improve >= cfg.patience:
            break
    
    if best_state:
        model.load_state_dict(best_state)
    return model

print("✓ MLP functions defined")

✓ MLP functions defined


## Step 8 — Train Neural Networks for Each Frequency

I train neural networks on all three frequencies (yearly, monthly, daily) to compare:

1. **Does more data help?** (yearly vs monthly vs daily)
2. **Can NNs beat the baseline?** (lag-based model)
3. **Which architecture works best?** (different layer sizes)

**Expected challenges:**
- Daily data has many points but may overfit due to interpolation smoothness
- Baseline is already very strong (R² > 0.99 for monthly/daily)
- NNs may struggle to add value when lag feature is nearly perfect

In [9]:
def train_and_evaluate_nn(data_dict, frequency_name, hidden_dims_list):
    """Train and evaluate neural network for a given frequency."""
    set_seed(SEED)
    
    X_train = data_dict["X_train"].values.astype(np.float64)
    y_train = data_dict["y_train"].values.astype(np.float64)
    X_test = data_dict["X_test"].values.astype(np.float64)
    y_test = data_dict["y_test"].values.astype(np.float64)
    
    # Remove any remaining NaN
    train_mask = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
    test_mask = ~(np.isnan(X_test).any(axis=1) | np.isnan(y_test))
    X_train = X_train[train_mask]
    y_train = y_train[train_mask]
    X_test = X_test[test_mask]
    y_test = y_test[test_mask]
    
    # Create validation set (last 20% of training data)
    n_val = max(1, int(len(X_train) * 0.2))
    X_train2 = X_train[:-n_val]
    y_train2 = y_train[:-n_val]
    X_val = X_train[-n_val:]
    y_val = y_train[-n_val:]
    
    # Standardize features
    X_train2_s, X_val_s, X_test_s = standardize_train_only(X_train2, X_val, X_test)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.tensor(X_train2_s, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train2, dtype=torch.float32, device=device)
    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test_s, dtype=torch.float32, device=device)
    
    cfg = TrainConfig()
    activation = nn.ELU()  # Exponential Linear Unit
    
    results = []
    for i, hidden_dims in enumerate(hidden_dims_list):
        arch_str = "x".join(map(str, hidden_dims))
        print(f"    Architecture {i+1}/{len(hidden_dims_list)}: {arch_str}", end=" ... ")
        t0_arch = time.time()
        
        model = MLPRegressor(X_train_t.shape[1], hidden_dims, activation).to(device)
        model = train_mlp(model, X_train_t, y_train_t, X_val_t, y_val_t, cfg, SEED)
        
        with torch.no_grad():
            model.eval()
            test_pred = model(X_test_t).cpu().numpy()
        
        # Check for NaN
        if np.isnan(test_pred).any():
            print(f"NaN predictions, skipping")
            continue
        
        r2 = r2_score(y_test, test_pred)
        print(f"R²={r2:.4f} ({time.time()-t0_arch:.1f}s)")
        
        results.append({
            "frequency": frequency_name,
            "architecture": arch_str,
            "r2": r2,
            "mse": mean_squared_error(y_test, test_pred),
            "mae": mean_absolute_error(y_test, test_pred),
        })
    
    return pd.DataFrame(results)

# Train NNs on all frequencies
print("Training Neural Networks...")
architectures = [(32,), (64,), (32, 16), (64, 32)]  # 4 architectures
nn_results = []

for name, data in [("yearly", yearly_data), ("monthly", monthly_data), ("daily", daily_data)]:
    print(f"\n  Training {name}...")
    t0 = time.time()
    results = train_and_evaluate_nn(data, name, architectures)
    print(f"    {name} training took {time.time()-t0:.1f}s")
    nn_results.append(results)

nn_results_df = pd.concat(nn_results, ignore_index=True)
print("\n📊 Neural Network Results:")
print(nn_results_df.to_string(index=False))

Training Neural Networks...

  Training yearly...
    Architecture 1/4: 32 ... R²=-3.7915 (0.8s)
    Architecture 2/4: 64 ... R²=-3.7672 (0.0s)
    Architecture 3/4: 32x16 ... R²=-3.5843 (0.0s)
    Architecture 4/4: 64x32 ... R²=-3.6107 (0.1s)
    yearly training took 1.0s

  Training monthly...
    Architecture 1/4: 32 ... R²=-3.8406 (0.0s)
    Architecture 2/4: 64 ... R²=-3.8658 (0.0s)
    Architecture 3/4: 32x16 ... R²=-3.6833 (0.1s)
    Architecture 4/4: 64x32 ... R²=-3.7186 (0.1s)
    monthly training took 0.2s

  Training daily...
    Architecture 1/4: 32 ... R²=-30.6303 (0.7s)
    Architecture 2/4: 64 ... R²=-152.6887 (0.9s)
    Architecture 3/4: 32x16 ... R²=-302.7052 (1.0s)
    Architecture 4/4: 64x32 ... R²=-161.0733 (11.6s)
    daily training took 14.3s

📊 Neural Network Results:
frequency architecture          r2          mse       mae
   yearly           32   -3.791482   808.810485 24.793595
   yearly           64   -3.767160   804.704811 24.797603
   yearly        32x16  

## Step 9 — Compare Results: Baseline vs Neural Networks

Now I compare the baseline (lag-based) model with the best neural network for each frequency.

**Key questions:**
1. Does more data (daily vs yearly) help neural networks?
2. Can neural networks beat the simple baseline?
3. What does this tell us about the value of interpolation?

In [10]:
# Combine baseline and NN results
comparison = []

# Add baselines
for _, row in baseline_df.iterrows():
    comparison.append({
        "frequency": row["frequency"],
        "model": "baseline",
        "architecture": "lag1",
        "r2": row["r2"],
        "mse": row["mse"],
        "mae": row["mae"],
    })

# Add best NN for each frequency
for freq in ["yearly", "monthly", "daily"]:
    freq_nn = nn_results_df[nn_results_df["frequency"] == freq]
    if len(freq_nn) > 0:
        best_nn = freq_nn.loc[freq_nn["r2"].idxmax()]
        comparison.append({
            "frequency": freq,
            "model": "nn",
            "architecture": best_nn["architecture"],
            "r2": best_nn["r2"],
            "mse": best_nn["mse"],
            "mae": best_nn["mae"],
        })

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.sort_values(["frequency", "r2"], ascending=[True, False])

print("="*70)
print("COMPARISON: Baseline vs Best Neural Network")
print("="*70)
print(comparison_df.to_string(index=False))

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for freq in ["yearly", "monthly", "daily"]:
    freq_data = comparison_df[comparison_df["frequency"] == freq]
    baseline_r2 = freq_data[freq_data["model"] == "baseline"]["r2"].values[0]
    nn_r2 = freq_data[freq_data["model"] == "nn"]["r2"].values[0] if len(freq_data[freq_data["model"] == "nn"]) > 0 else None
    
    print(f"\n{freq.upper()}:")
    print(f"  Baseline R²: {baseline_r2:.4f}")
    if nn_r2 is not None:
        print(f"  Best NN R²:  {nn_r2:.4f}")
        diff = nn_r2 - baseline_r2
        print(f"  Difference:  {diff:+.4f} ({'NN wins!' if nn_r2 > baseline_r2 else 'Baseline wins'})")

COMPARISON: Baseline vs Best Neural Network
frequency    model architecture         r2         mse       mae
    daily baseline         lag1   0.999999    0.000166  0.007267
    daily       nn           32 -30.630328 4894.416448 40.236837
  monthly baseline         lag1   0.998961    0.161423  0.229230
  monthly       nn        32x16  -3.683276  727.772917 23.428752
   yearly baseline         lag1   0.727857   45.938182  5.109091
   yearly       nn        32x16  -3.584276  773.833739 24.310891

SUMMARY

YEARLY:
  Baseline R²: 0.7279
  Best NN R²:  -3.5843
  Difference:  -4.3121 (Baseline wins)

MONTHLY:
  Baseline R²: 0.9990
  Best NN R²:  -3.6833
  Difference:  -4.6822 (Baseline wins)

DAILY:
  Baseline R²: 1.0000
  Best NN R²:  -30.6303
  Difference:  -31.6303 (Baseline wins)


## Step 10 — Save Results

I save all the augmented datasets and results for future analysis.

In [11]:
# Save augmented datasets
os.makedirs("../data/processed/panel", exist_ok=True)
os.makedirs("../data/processed/results", exist_ok=True)

df_monthly.to_csv("../data/processed/panel/marz_monthly_panel_augmented.csv", index=False)
df_daily.to_csv("../data/processed/panel/marz_daily_panel_augmented.csv", index=False)
print("✓ Saved augmented datasets")

# Save results
comparison_df.to_csv("../data/processed/results/augmentation_forecasting_comparison.csv", index=False)
nn_results_df.to_csv("../data/processed/results/augmentation_nn_results.csv", index=False)
baseline_df.to_csv("../data/processed/results/augmentation_baseline_results.csv", index=False)
print("✓ Saved results")

print("\n🎉 Analysis complete!")

✓ Saved augmented datasets
✓ Saved results

🎉 Analysis complete!


## Key Takeaways & Interpretation

### What I Learned

1. **Interpolation creates smooth data**: The high R² scores (0.99+) for monthly/daily baselines are largely due to interpolation smoothness, not real predictive power.

2. **More data doesn't always help**: Having 27,797 daily points vs 77 yearly points didn't help neural networks beat the baseline. In fact, NNs performed worse on daily data.

3. **Simple baselines are strong**: The lag-based baseline is hard to beat, especially with interpolated data where lag features are nearly perfect.

4. **Interpolation artifacts**: The artificial smoothness from linear interpolation makes prediction trivial (lag ≈ target), which doesn't reflect real-world forecasting challenges.

### Limitations of This Approach

- **Artificial smoothness**: Real monthly/daily data would have more noise and volatility
- **No new information**: Interpolation doesn't add new signal, just creates intermediate points
- **Overfitting risk**: Models may learn interpolation patterns rather than real relationships

This analysis was valuable for understanding the limits of interpolation-based augmentation and confirming that simple baselines can be very strong in time series forecasting.